# DOTA Pipeline (Kaggle)

Kaggle-версия пайплайна для DOTA:
- читает YOLO-датасет из `/kaggle/input`
- выполняет smoke + full pipeline
- сохраняет все результаты в `/kaggle/working/outputs/dota`


In [ ]:
# Step 0: workspace bootstrap.
from pathlib import Path
import os
import sys
import subprocess

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_ROOT = KAGGLE_WORKING / 'small_objects_auto_aug'

REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'
ALLOW_GIT_CLONE = True

if not PROJECT_ROOT.exists():
    if ALLOW_GIT_CLONE:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        raise FileNotFoundError(f'Project repo is missing at {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('KAGGLE_INPUT exists =', KAGGLE_INPUT.exists())

In [ ]:
# Step 1: install dependencies.
import sys
import subprocess
from pathlib import Path

SKIP_INSTALL = False

if not SKIP_INSTALL:
    req = Path('requirements.txt')
    if req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)
    else:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '-q',
            'ultralytics', 'albumentations', 'opencv-python-headless', 'pycocotools',
            'tqdm', 'pyyaml', 'matplotlib', 'pandas'
        ], check=True)
    print('[OK] Dependencies are installed.')
else:
    print('[SKIP] Dependency installation is disabled.')

In [ ]:
# Step 2: locate dataset in /kaggle/input and prepare runtime paths.
from pathlib import Path
import shutil

DATASET_SLUG_HINTS = ['small-objects-dota-yolo', 'dota', 'dota-yolo', 'dotav1']
USER_DATASET_ROOT = None  # e.g. Path('/kaggle/input/my-dota-yolo')
USE_LOCAL_RUNTIME_COPY = True
LOCAL_RUNTIME_ROOT = Path('/kaggle/working/datasets/dota_yolo')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

OUTPUT_ROOT = Path('/kaggle/working/outputs/dota')
DOTA_EXPERIMENT_RUNS = OUTPUT_ROOT / 'runs'
DOTA_EXPERIMENT_ARTIFACTS = OUTPUT_ROOT / 'artifacts'
DOTA_EXPERIMENT_RUNS.mkdir(parents=True, exist_ok=True)
DOTA_EXPERIMENT_ARTIFACTS.mkdir(parents=True, exist_ok=True)


def _count_images(images_dir: Path) -> int:
    if not images_dir.exists():
        return 0
    return sum(1 for p in images_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def _is_ready_yolo(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in req):
        return False
    return _count_images(root / 'images' / 'train') > 0 and _count_images(root / 'images' / 'val') > 0


def _find_yolo_root() -> Path | None:
    candidates = []
    if USER_DATASET_ROOT is not None:
        candidates.append(Path(USER_DATASET_ROOT))

    for slug in DATASET_SLUG_HINTS:
        candidates.append(Path('/kaggle/input') / slug)

    for entry in sorted(Path('/kaggle/input').iterdir()) if Path('/kaggle/input').exists() else []:
        candidates.append(entry)

    seen = set()
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        if c.exists() and _is_ready_yolo(c):
            return c

        nested = c / 'dota_yolo'
        if nested.exists() and _is_ready_yolo(nested):
            return nested

        nested2 = c / 'datasets' / 'dota_yolo'
        if nested2.exists() and _is_ready_yolo(nested2):
            return nested2

    for images_train in Path('/kaggle/input').rglob('images/train') if Path('/kaggle/input').exists() else []:
        root = images_train.parent.parent
        if _is_ready_yolo(root):
            return root
    return None


found_root = _find_yolo_root()
if found_root is None:
    preview = [p.name for p in sorted(Path('/kaggle/input').iterdir())] if Path('/kaggle/input').exists() else []
    raise FileNotFoundError(
        'YOLO DOTA dataset not found in /kaggle/input. '
        f'Available inputs: {preview}. '
        'Expected structure: images/{train,val} and labels/{train,val}.'
    )

TARGET_DATASET_ROOT = found_root
print('[OK] Found dataset:', TARGET_DATASET_ROOT)

if USE_LOCAL_RUNTIME_COPY:
    if not _is_ready_yolo(LOCAL_RUNTIME_ROOT):
        if LOCAL_RUNTIME_ROOT.exists():
            shutil.rmtree(LOCAL_RUNTIME_ROOT)
        print(f'[INFO] Copy dataset to local SSD: {TARGET_DATASET_ROOT} -> {LOCAL_RUNTIME_ROOT}')
        shutil.copytree(TARGET_DATASET_ROOT, LOCAL_RUNTIME_ROOT)
    else:
        print('[OK] Reuse local runtime dataset:', LOCAL_RUNTIME_ROOT)
    TARGET_DATASET_ROOT = LOCAL_RUNTIME_ROOT

print('[OK] Runtime dataset root:', TARGET_DATASET_ROOT)
print('train images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'train'))
print('val images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'val'))
print('runs root:', DOTA_EXPERIMENT_RUNS)
print('artifacts root:', DOTA_EXPERIMENT_ARTIFACTS)

In [ ]:
# Step 3: build runtime config from template.
from pathlib import Path
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'dota_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / 'dota_runtime_config_kaggle.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg.setdefault('dataset', {})
cfg.setdefault('training', {})

cfg['dataset']['kind'] = 'yolo_generic'
cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(TARGET_DATASET_ROOT)

possible_yaml = [
    TARGET_DATASET_ROOT / 'dataset.yaml',
    TARGET_DATASET_ROOT / 'data.yaml',
    TARGET_DATASET_ROOT / 'dota.yaml',
]
data_yaml_path = next((p for p in possible_yaml if p.exists()), None)
if data_yaml_path is None:
    runtime_data_yaml = PROJECT_ROOT / 'artifacts' / 'runtime_dota_data_kaggle.yaml'
    class_names = cfg['dataset'].get('class_names', [str(i) for i in range(15)])
    payload = {
        'path': str(TARGET_DATASET_ROOT),
        'train': 'images/train',
        'val': 'images/val',
        'names': {i: name for i, name in enumerate(class_names)},
    }
    runtime_data_yaml.write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding='utf-8')
    data_yaml_path = runtime_data_yaml

cfg['training']['project_dir'] = str(DOTA_EXPERIMENT_RUNS)
cfg['training']['data_yaml'] = str(data_yaml_path)
cfg['training']['run_ablation'] = False
cfg['training']['cache'] = 'disk'
cfg['training']['val_during_train'] = True

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('Runtime config:', runtime_cfg_path)
print('Data yaml:', data_yaml_path)
print('Runs dir:', cfg['training']['project_dir'])

In [ ]:
# Step 4: smoke run (subset -> analyze -> policy).
from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import load_yaml

cfg = load_yaml(runtime_cfg_path)

subset_root = PROJECT_ROOT / 'datasets' / 'dota_smoke_kaggle'
smoke_stats_dir = PROJECT_ROOT / 'artifacts' / 'dota_smoke' / 'stats'
smoke_policy_dir = PROJECT_ROOT / 'artifacts' / 'dota_smoke' / 'policy'

subset_report = build_yolo_subset(
    dataset_root=TARGET_DATASET_ROOT,
    output_root=subset_root,
    train_images=120,
    val_images=40,
    seed=42,
    clean_output=True,
)
print('subset report:', subset_report)

stats = analyze_dataset(
    dataset_root=subset_root,
    output_dir=smoke_stats_dir,
    splits=('train',),
    config=DatasetAnalyzerConfig(
        small_max_area=float(cfg['analysis']['small_max_area']),
        medium_max_area=float(cfg['analysis']['medium_max_area']),
        tiny_max_area=float(cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
        show_progress=True,
    ),
)

policy, decision_report = generate_policy_from_stats(
    stats,
    cfg=RuleEngineConfig.from_project_config(cfg),
)
paths = save_policy_artifacts(policy, decision_report, output_dir=smoke_policy_dir)

print('[OK] smoke done')
print('stats:', smoke_stats_dir / 'dataset_stats.json')
print('policy:', paths['policy_json'])

In [ ]:
# Step 5: full pipeline run with execution profiles.
from pathlib import Path
import yaml
from src.pipeline_mvp import run_mvp_pipeline

assert (PROJECT_ROOT / 'configs' / 'baseline.yaml').exists(), 'Missing configs/baseline.yaml'
assert (PROJECT_ROOT / 'configs' / 'manual.yaml').exists(), 'Missing configs/manual.yaml'

EXECUTION_PROFILE = 'hour_safe'  # fast | balanced | hour_safe | hour_full | max_quality

profile_cfg = {
    'fast': {
        'run_training': True,
        'run_eval': False,
        'train_profile': 'fast',
        'run_ablation': False,
        'auto_predict_for_eval': False,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 8,
        'mode_overrides': {
            'baseline': {'epochs': 12, 'fraction': 0.5, 'imgsz': 768, 'val': True},
            'manual': {'epochs': 12, 'fraction': 0.5, 'imgsz': 768, 'val': True},
            'adaptive': {'epochs': 12, 'fraction': 0.5, 'imgsz': 768, 'val': True},
        },
    },
    'balanced': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'balanced',
        'run_ablation': False,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 10,
        'mode_overrides': {
            'baseline': {'epochs': 20, 'fraction': 0.8, 'imgsz': 896, 'val': True},
            'manual': {'epochs': 20, 'fraction': 0.8, 'imgsz': 896, 'val': True},
            'adaptive': {'epochs': 20, 'fraction': 0.8, 'imgsz': 896, 'val': True},
        },
    },
    'hour_safe': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'hour',
        'run_ablation': False,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 12,
        'mode_overrides': {
            'baseline': {'epochs': 25, 'fraction': 1.0, 'imgsz': 960, 'val': True},
            'manual': {'epochs': 25, 'fraction': 1.0, 'imgsz': 960, 'val': True},
            'adaptive': {'epochs': 25, 'fraction': 1.0, 'imgsz': 960, 'val': True},
        },
    },
    'hour_full': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'hour',
        'run_ablation': True,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 18,
        'mode_overrides': {
            'baseline': {'epochs': 30, 'fraction': 1.0, 'imgsz': 960, 'val': True},
            'manual': {'epochs': 30, 'fraction': 1.0, 'imgsz': 960, 'val': True},
            'adaptive': {'epochs': 30, 'fraction': 1.0, 'imgsz': 960, 'val': True},
        },
    },
    'max_quality': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'max_quality',
        'run_ablation': True,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 24,
        'mode_overrides': {
            'baseline': {'epochs': 40, 'fraction': 1.0, 'imgsz': 1024, 'val': True},
            'manual': {'epochs': 40, 'fraction': 1.0, 'imgsz': 1024, 'val': True},
            'adaptive': {'epochs': 40, 'fraction': 1.0, 'imgsz': 1024, 'val': True},
        },
    },
}

assert EXECUTION_PROFILE in profile_cfg, f'Unknown EXECUTION_PROFILE: {EXECUTION_PROFILE}'
selected = profile_cfg[EXECUTION_PROFILE]

with runtime_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg.setdefault('dataset', {})
cfg.setdefault('training', {})
cfg['dataset']['root'] = str(TARGET_DATASET_ROOT)
cfg['training']['run_ablation'] = bool(selected['run_ablation'])
cfg['training']['project_dir'] = str(DOTA_EXPERIMENT_RUNS)
cfg['training']['val_during_train'] = bool(selected['val_during_train'])
cfg['training']['cache'] = selected['cache']
cfg['training']['patience'] = int(selected['patience'])
cfg['training']['mode_overrides'] = selected['mode_overrides']

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f'[INFO] EXECUTION_PROFILE={EXECUTION_PROFILE}')
print(f'[INFO] training.project_dir={cfg["training"]["project_dir"]}')
print(f'[INFO] dataset.root={cfg["dataset"]["root"]}')

outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=bool(selected['run_training']),
    run_eval=bool(selected['run_eval']),
    train_profile=str(selected['train_profile']),
    auto_predict_for_eval=bool(selected['auto_predict_for_eval']),
)
print(outputs)

In [ ]:
# Step 6: unified monitoring export (train + val metrics history).
from pathlib import Path
import pandas as pd

PROJECT_ROOT_RESOLVED = Path(globals().get('PROJECT_ROOT', Path.cwd()))
RUNS_ROOT = Path(globals().get('DOTA_EXPERIMENT_RUNS', PROJECT_ROOT_RESOLVED / 'runs'))

monitor_dir = PROJECT_ROOT_RESOLVED / 'artifacts' / 'monitoring'
monitor_dir.mkdir(parents=True, exist_ok=True)

results_files = sorted(RUNS_ROOT.rglob('results.csv'))
if not results_files:
    raise FileNotFoundError(f'No results.csv found under runs root: {RUNS_ROOT}')

frames = []
for csv_path in results_files:
    run_dir = csv_path.parent
    run_name = run_dir.name
    df = pd.read_csv(csv_path)
    if df.empty:
        continue
    if 'epoch' not in df.columns:
        df['epoch'] = list(range(len(df)))
    df['run_name'] = run_name
    df['run_dir'] = run_dir.as_posix()
    frames.append(df)

if not frames:
    raise RuntimeError('No readable non-empty results.csv files were found.')

history_wide = pd.concat(frames, ignore_index=True)
id_cols = ['run_name', 'run_dir', 'epoch']
metric_cols = [c for c in history_wide.columns if c not in id_cols and pd.api.types.is_numeric_dtype(history_wide[c])]
history_wide = history_wide[id_cols + metric_cols]

history_wide_path = monitor_dir / 'metrics_history_wide.csv'
history_wide.to_csv(history_wide_path, index=False)

last_rows = history_wide.sort_values(['run_name', 'epoch']).groupby('run_name', as_index=False).tail(1)
last_rows = last_rows.sort_values('run_name').reset_index(drop=True)
last_rows_path = monitor_dir / 'metrics_last_epoch.csv'
last_rows.to_csv(last_rows_path, index=False)

history_long = history_wide.melt(
    id_vars=['run_name', 'run_dir', 'epoch'],
    value_vars=metric_cols,
    var_name='metric',
    value_name='value',
)
history_long['split'] = history_long['metric'].map(
    lambda m: 'val' if str(m).lower().startswith(('metrics/', 'val/')) else ('train' if 'loss' in str(m).lower() else 'other')
)
history_long_path = monitor_dir / 'metrics_history_long.csv'
history_long.to_csv(history_long_path, index=False)

print('[OK] history_wide:', history_wide_path)
print('[OK] history_long:', history_long_path)
print('[OK] last_epoch:', last_rows_path)
print(last_rows[['run_name', 'epoch'] + [c for c in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)'] if c in last_rows.columns]].to_string(index=False))

In [ ]:
# Step 7: plot preview for train/val metrics.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RUN_PLOT_PREVIEW = True

if RUN_PLOT_PREVIEW:
    monitor_dir = Path(PROJECT_ROOT) / 'artifacts' / 'monitoring'
    history_long_path = monitor_dir / 'metrics_history_long.csv'
    if not history_long_path.exists():
        raise FileNotFoundError(f'Missing file: {history_long_path}')

    df = pd.read_csv(history_long_path)
    df = df[pd.to_numeric(df['value'], errors='coerce').notna()].copy()
    df['value'] = df['value'].astype(float)

    plots_dir = monitor_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    metric_candidates = [
        'metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)',
        'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
        'box_loss', 'cls_loss', 'dfl_loss',
    ]
    metrics = [m for m in metric_candidates if m in set(df['metric'].unique())]

    created = []
    for metric in metrics:
        sub = df[df['metric'] == metric].copy()
        if sub.empty:
            continue
        plt.figure(figsize=(9, 5))
        for run_name, grp in sub.groupby('run_name'):
            grp = grp.sort_values('epoch')
            plt.plot(grp['epoch'], grp['value'], marker='o', markersize=2, linewidth=1.6, label=str(run_name))
        plt.title(f'{metric} vs epoch')
        plt.xlabel('epoch')
        plt.ylabel(metric)
        plt.grid(alpha=0.3)
        plt.legend(loc='best', fontsize=8)
        out = plots_dir / f"metric_{metric.replace('/', '_').replace('(', '').replace(')', '')}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        created.append(out)

    print('[OK] Plot files created:')
    for p in created:
        print(' -', p)
else:
    print('Plot preview skipped.')

In [ ]:
# Step 8: pack outputs for Kaggle output panel download.
from pathlib import Path
import shutil

OUTPUT_ROOT = Path('/kaggle/working/outputs/dota')
ARCHIVE_BASE = Path('/kaggle/working/dota_outputs')

if OUTPUT_ROOT.exists():
    archive_path = shutil.make_archive(str(ARCHIVE_BASE), 'zip', root_dir=OUTPUT_ROOT)
    print('[OK] Created archive:', archive_path)
else:
    print('[WARN] Output root does not exist yet:', OUTPUT_ROOT)